# 🏨 Ensembles and optimisation: predicting hotel cancellations
### From a leak-free model to a stored artefact that an app can use

**Module 1 · Session 10 · lecture notebook**

Session 09 ended with a checklist: beat a baseline, measure on unseen data, keep the test set out of every
choice, ask whether each feature exists at prediction time, and choose thresholds from what mistakes cost.
This notebook applies all of it to a real business problem, and adds four things:

1. **Time.** When data arrives over time, the split has to respect it.
2. **Ensembles.** Many trees together: random forests (bagging) and gradient boosting.
3. **Optimisation.** Tuning hyperparameters with Optuna, on validation data.
4. **Trustworthy probabilities.** Calibration, cost-based thresholds, and forecasts built from probabilities.

It ends with the model stored properly, ready for the Streamlit app *Tonight's front desk*.

Run all cells (Runtime → Run all). Plotting code is folded away (*Show code*).

Contents
1. The problem and the data
2. A split that respects time
3. Leakage hunt
4. Ensembles: bagging and boosting
5. Optimisation with Optuna
6. Can the probabilities be trusted?
7. From probabilities to decisions and forecasts
8. What drives a cancellation?
9. Storing the model properly

In [ ]:
# @title Setup: libraries and chart style { display-mode: "form" }
%pip install -q optuna shap
import json, time, warnings, platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, xgboost as xgb, optuna, shap
warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
print("scikit-learn", sklearn.__version__, "· xgboost", xgb.__version__, "· pandas", pd.__version__)

---
# 🏨 1 · The problem and the data

Two hotels in Portugal, a city hotel in Lisbon and a resort in the Algarve, 119,390 bookings from July 2015
to August 2017 ([Antonio, Almeida & Nunes, 2019](https://doi.org/10.1016/j.dib.2018.11.126)). About one
booking in three is cancelled.

A cancellation is expensive when it comes as a surprise: the room stays empty. Hotels respond by
**overbooking**, selling more rooms than they have. Overbook too little and rooms stay empty; too much and
guests who arrive are "walked" to another hotel, at the hotel's expense. Knowing *which* bookings are likely
to cancel, and *how many* will cancel on a given night, is worth real money.

🤔 *Question: what would the hotel need to know about a booking, at the moment it is made, to judge whether
it will cancel?*

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv")
df["arrival_date"] = pd.to_datetime(df["arrival_date_year"].astype(str) + "-" + df["arrival_date_month"]
                                    + "-" + df["arrival_date_day_of_month"].astype(str))
print(df.shape, "·", df["arrival_date"].min().date(), "to", df["arrival_date"].max().date())
print(f"cancelled: {df['is_canceled'].mean():.0%}")
df[["hotel", "lead_time", "adr", "deposit_type", "market_segment", "is_canceled"]].describe(include="all").T.iloc[:, :6]

Two values need a decision before anything else (the session 03 rule: look before you trust):
`adr`, the average daily rate, has one booking at 5,400 EUR a night and one below zero, and `children` has
four missing values.

In [ ]:
df = df[df["adr"].between(0, 1000)].copy()          # drop the two impossible prices
df["children"] = df["children"].fillna(0)

# A few features built from what is known when the booking is made
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["arrival_weekday"] = df["arrival_date"].dt.dayofweek
df["agent"] = df["agent"].fillna(0).astype(int).astype(str)     # an ID is a category, not a number
print(df.shape)

---
# 🕰️ 2 · A split that respects time

In session 09 the split was random, which is fine when rows do not depend on time. Bookings do. A model
is trained on the past and used on the future, and prices, customers and policies drift in between. A random
split lets the model learn from next summer to predict last winter.

So the split follows the calendar:
- **train**: arrivals up to November 2016
- **validation**: December 2016 to March 2017, for choosing models, settings and thresholds
- **test**: April to August 2017, used once at the end

In [ ]:
NUM = ["lead_time", "total_nights", "stays_in_weekend_nights", "adults", "children", "babies",
       "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled", "adr",
       "arrival_weekday", "total_guests"]
CAT = ["hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type", "deposit_type",
       "customer_type", "agent"]

train = df["arrival_date"] < "2016-12-01"
val = (df["arrival_date"] >= "2016-12-01") & (df["arrival_date"] < "2017-04-01")
test = df["arrival_date"] >= "2017-04-01"
y = df["is_canceled"]
for name, part in [("train", train), ("validation", val), ("test", test)]:
    print(f"{name:10s} {part.sum():6,d} bookings · cancelled {y[part].mean():.0%}")

In [ ]:
# @title Draw the split { display-mode: "form" }
monthly = df.groupby([df["arrival_date"].dt.to_period("M"), np.select([train, val], ["train", "validation"], "test")]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 3))
bottom = np.zeros(len(monthly))
for part, colour in [("train", NAVY), ("validation", LAV), ("test", ORANGE)]:
    ax.bar(monthly.index.astype(str), monthly[part], bottom=bottom, color=colour, label=part); bottom += monthly[part].values
ax.set(ylabel="bookings", title="Bookings per arrival month"); ax.legend(ncol=3)
ax.set_xticks(range(0, len(monthly), 3)); ax.set_xticklabels(monthly.index.astype(str)[::3], rotation=0, fontsize=9)
plt.show()

Notice the test period cancels more often than the training period. The world moved. A model trained on the
past will be a little off on the future, and we will see that again when the forecasts come in.

---
# 🚰 3 · Leakage hunt

The table has more columns than the ones chosen above. Here is a model that uses six of them as well.

In [ ]:
def preprocessor(num, cat):
    # numbers: fill gaps, scale · categories: one column per level, rare levels pooled, unseen levels tolerated
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20, sparse_output=False), cat)])

def fit_and_score(model, num=NUM, cat=CAT):
    pipe = make_pipeline(preprocessor(num, cat), model).fit(df.loc[train, num + cat], y[train])
    p = pipe.predict_proba(df.loc[val, num + cat])[:, 1]
    return pipe, p, roc_auc_score(y[val], p)

df["room_changed"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)
df["is_portugal"] = (df["country"] == "PRT").astype(int)
EXTRA = ["required_car_parking_spaces", "total_of_special_requests", "booking_changes",
         "days_in_waiting_list", "room_changed", "is_portugal"]

booster = lambda: xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                                    colsample_bytree=0.8, tree_method="hist", random_state=0)
_, _, auc_all = fit_and_score(booster(), NUM + EXTRA)
_, _, auc_clean = fit_and_score(booster())
print(f"with the six extra columns: AUC {auc_all:.3f}")
print(f"without them:               AUC {auc_clean:.3f}")

The extra columns add a lot. 🤔 *Question: which of them would the hotel know on the day the booking is made?*

In [ ]:
print("cancellation rate by column value:\n")
print("parking space requested:  ", df.groupby(df["required_car_parking_spaces"] > 0)["is_canceled"].mean().round(3).to_dict())
print("assigned room ≠ reserved: ", df.groupby("room_changed")["is_canceled"].mean().round(3).to_dict())
print("country = Portugal:       ", df.groupby("is_portugal")["is_canceled"].mean().round(3).to_dict())

- **Parking**: not a single booking with a parking space was cancelled. Parking is recorded when the guest
  arrives, so a guest who never arrives never has one. The column contains the answer.
- **Assigned room**: the room is assigned at check-in. A different room means the guest turned up.
- **Country**: for many bookings the nationality is only confirmed at check-in; cancelled bookings keep a
  default. The dataset's authors describe this.
- **Special requests, booking changes, waiting list**: these keep being updated until arrival. Their value at
  booking time is not what the table shows.

All six are dropped. The honest AUC is lower, and it is the one that will hold on the day the model is used.
This is session 09's leakage question on a real dataset: *does this column exist at the moment of prediction?*

---
# 🌲 4 · Ensembles: many trees instead of one

The ladder from session 09, on the validation months: a baseline, a linear model, one tree, and two
**ensembles**, models built from many trees.

In [ ]:
ladder = {"baseline (always the average)": DummyClassifier(strategy="prior"),
          "logistic regression": LogisticRegression(max_iter=2000),
          "one deep tree": DecisionTreeClassifier(random_state=0),
          "random forest (bagging)": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=0),
          "XGBoost, default settings": xgb.XGBClassifier(tree_method="hist", random_state=0),
          "XGBoost, slower learning": booster()}
rows, val_proba = [], {}
for name, model in ladder.items():
    t0 = time.time()
    _, p, auc = fit_and_score(model)
    val_proba[name] = p
    rows.append((name, auc, log_loss(y[val], np.clip(p, 1e-6, 1 - 1e-6)), time.time() - t0))
pd.DataFrame(rows, columns=["model", "AUC", "log loss", "seconds"]).set_index("model").round(3)

Higher AUC is better (ranking), lower log loss is better (probabilities). One deep tree is the worst real model:
it memorises. Hundreds of trees together are among the best. How can many bad models make a good one?
Two different ideas.

### 🧭 Side quest: bagging, drawn

**Bagging** (bootstrap aggregating) trains many deep trees, each on a random resample of the rows, and averages
them. Each tree overfits in its own way; the averaging cancels most of that out. It is the variance side of
session 09's bias–variance picture. A random forest adds one more trick: each split may only look at a random
subset of the columns, so the trees differ even more.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 10, 120)); truth = np.sin(x) + 0.3 * x
yy = truth + rng.normal(0, 0.6, len(x)); grid = np.linspace(0, 10, 400)
trees = []
for _ in range(200):
    i = rng.integers(0, len(x), len(x))                                   # a bootstrap resample
    trees.append(DecisionTreeRegressor(min_samples_leaf=3).fit(x[i, None], yy[i]).predict(grid[:, None]))
trees = np.array(trees)
print(f"spread of single trees: {trees.std(axis=0).mean():.2f} · error of one tree vs truth: "
      f"{np.abs(trees[0] - (np.sin(grid) + 0.3 * grid)).mean():.2f} · of the average: {np.abs(trees.mean(0) - (np.sin(grid) + 0.3 * grid)).mean():.2f}")

In [ ]:
# @title Draw bagging { display-mode: "form" }
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for t in trees[:5]: a.plot(grid, t, color=ORANGE, lw=1, alpha=.6)
a.scatter(x, yy, s=10, color=LAV); a.set(title="5 deep trees, 5 resamples: each one overfits")
b.scatter(x, yy, s=10, color=LAV); b.plot(grid, trees.mean(0), color=NAVY, lw=2.5, label="average of 200 trees")
b.plot(grid, np.sin(grid) + 0.3 * grid, color=GREY, ls="--", lw=1.5, label="true pattern")
b.set(title="Their average: smooth and close to the truth"); b.legend()
plt.tight_layout(); plt.show()

### 🧭 Side quest: boosting, drawn

**Boosting** works the other way round. It starts with a very simple model and adds small trees one after
another, each one fitted to the errors that are still left. The learning rate decides how much of each new
tree is added: small steps need more trees, but generalise better, which is why "slower learning" edged out the
default settings in the ladder above. Too many trees overfit again: in the last panel the curve starts to
follow the noise. XGBoost is a fast, regularised implementation of this idea.

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2).fit(x[:, None], yy)
stages = list(gb.staged_predict(grid[:, None]))
for n in [1, 5, 20, 100]:
    print(f"after {n:3d} small trees: training error {np.abs(list(gb.staged_predict(x[:, None]))[n - 1] - yy).mean():.2f}")

In [ ]:
# @title Draw boosting { display-mode: "form" }
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True)
for ax, n in zip(axes, [1, 5, 20, 100]):
    ax.scatter(x, yy, s=8, color=LAV); ax.plot(grid, stages[n - 1], color=ORANGE, lw=2)
    ax.set(title=f"{n} tree{'s' if n > 1 else ''}")
plt.suptitle("Boosting: each new small tree fixes part of what is still wrong", y=1.03); plt.tight_layout(); plt.show()

| | Bagging (random forest) | Boosting (XGBoost, LightGBM, CatBoost) |
|---|---|---|
| Trees | deep, independent, in parallel | shallow, in sequence, each fixing the last |
| Mainly reduces | variance | bias (and variance, with a small learning rate) |
| Tuning | forgiving | matters: learning rate, depth, number of trees |
| On business tables | a strong default | usually the best, once tuned |

"Once tuned" is the next section.

---
# 🎛️ 5 · Optimisation with Optuna

XGBoost has many settings (hyperparameters). Grid search, from session 09, tries every combination, which gets
expensive fast. **Optuna** samples settings, keeps track of which regions scored well, and samples more there.

Two rules from session 09 still apply: tuning uses the **validation** months, never the test months, and the
score is a proper one. Log loss rewards good probabilities, which the forecasts later depend on. Each trial
also uses **early stopping**: trees are added until the validation score stops improving.

In [ ]:
prep = preprocessor(NUM, CAT).fit(df.loc[train, NUM + CAT])
Xtr, Xva = prep.transform(df.loc[train, NUM + CAT]), prep.transform(df.loc[val, NUM + CAT])

def objective(trial):
    params = {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
              "max_depth": trial.suggest_int("max_depth", 3, 10),
              "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
              "subsample": trial.suggest_float("subsample", 0.5, 1.0),
              "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
              "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20, log=True)}
    model = xgb.XGBClassifier(n_estimators=2000, early_stopping_rounds=50, eval_metric="logloss",
                              tree_method="hist", random_state=0, **params)
    model.fit(Xtr, y[train], eval_set=[(Xva, y[val])], verbose=False)
    trial.set_user_attr("n_trees", model.best_iteration + 1)
    return model.best_score                                   # validation log loss at the best number of trees

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=0))
study.optimize(objective, n_trials=25, timeout=240)
best = study.best_trial
print(f"{len(study.trials)} trials · best validation log loss {best.value:.4f} with {best.user_attrs['n_trees']} trees")
print({k: round(v, 3) for k, v in best.params.items()})

In [ ]:
# @title Draw the search { display-mode: "form" }
hist = study.trials_dataframe()
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.scatter(hist["number"], hist["value"], color=LAV, s=25); a.plot(hist["number"], hist["value"].cummin(), color=ORANGE, lw=2)
a.set(xlabel="trial", ylabel="validation log loss", title="Best so far (orange)")
b.scatter(hist["params_learning_rate"], hist["value"], c=hist["params_max_depth"], cmap="viridis", s=30)
b.set(xscale="log", xlabel="learning rate", ylabel="validation log loss", title="Each trial (colour: depth)")
plt.tight_layout(); plt.show()

On this data the search finds little to gain: many settings land within a hair of each other, and the defaults
were already close. That is common for tables with a few strong signals, and worth knowing: tuning is not
magic, and it is cheap to check. The number of trees, set by early stopping, matters most.

The tuned settings are then used for the final model, trained on train **and** validation months with the
number of trees Optuna found. The test months are still untouched.

In [ ]:
FEATURES = NUM + CAT
trainval = train | val
final = make_pipeline(preprocessor(NUM, CAT),
                      xgb.XGBClassifier(n_estimators=best.user_attrs["n_trees"], tree_method="hist",
                                        random_state=0, **best.params))
final.fit(df.loc[trainval, FEATURES], y[trainval])
p_test = final.predict_proba(df.loc[test, FEATURES])[:, 1]
print(f"test AUC {roc_auc_score(y[test], p_test):.3f} · test log loss {log_loss(y[test], p_test):.3f}   (test months, used once)")

---
# 📐 6 · Can the probabilities be trusted?

AUC only checks the **ranking**: are cancellations scored above non-cancellations? For decisions and forecasts
the numbers themselves have to mean something. A model is **calibrated** if, of all bookings it gives 30 %,
about 30 % actually cancel.

A popular trick for unbalanced classes, `scale_pos_weight` (or `class_weight="balanced"`), makes the model
pay more attention to cancellations. It also inflates every probability. Compare, on the validation months:

In [ ]:
_, p_weighted, _ = fit_and_score(xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                                                   colsample_bytree=0.8, tree_method="hist", random_state=0,
                                                   scale_pos_weight=(y[train] == 0).sum() / (y[train] == 1).sum()))
curves = {"XGBoost": val_proba["XGBoost, slower learning"], "XGBoost, class-weighted": p_weighted,
          "random forest": val_proba["random forest (bagging)"], "logistic regression": val_proba["logistic regression"]}
print(f"actual cancellation rate in validation: {y[val].mean():.3f}\n")
for name, p in curves.items():
    print(f"{name:25s} mean predicted {p.mean():.3f} · Brier score {brier_score_loss(y[val], p):.3f}")

In [ ]:
# @title Draw the reliability diagram { display-mode: "form" }
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1, label="perfectly calibrated")
for (name, p), colour in zip(curves.items(), [NAVY, ORANGE, "#6C8EBF", LAV]):
    frac, mean_p = calibration_curve(y[val], p, n_bins=10, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", ms=4, color=colour, label=name)
ax.set(xlabel="predicted probability", ylabel="actual share cancelled", title="Reliability diagram (validation)")
ax.legend(fontsize=9); plt.show()

The class-weighted model sits below the diagonal: where it says 40 %, under 30 % cancel. Its ranking (AUC) is
fine, but its probabilities cannot be added up or compared with costs. The unweighted models are close to the
diagonal, slightly above it: the validation winter cancelled a little more than the training months predicted. If a model is miscalibrated, `CalibratedClassifierCV` can repair it on held-out data; here it is
simpler not to distort it in the first place.

---
# 💶 7 · From probabilities to decisions and forecasts

### A threshold from costs
Say the hotel phones every booking flagged as risky to ask for reconfirmation. An unnecessary call costs about
5 EUR of staff time and a little goodwill. A cancellation found in time lets the room be resold, worth about
40 EUR on average. As in session 09, the cut-off is chosen on the **validation** months and checked once on test.

In [ ]:
COST_CALL, VALUE_CATCH = 5, 40
def net_value(truth, p, t):
    flagged = p >= t
    return VALUE_CATCH * (flagged & (truth == 1)).sum() - COST_CALL * flagged.sum()

p_val_final = make_pipeline(preprocessor(NUM, CAT), xgb.XGBClassifier(
    n_estimators=best.user_attrs["n_trees"], tree_method="hist", random_state=0, **best.params)
).fit(df.loc[train, FEATURES], y[train]).predict_proba(df.loc[val, FEATURES])[:, 1]
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
values = [net_value(y[val].values, p_val_final, t) for t in thresholds]
t_star = float(thresholds[int(np.argmax(values))])
print(f"best cut-off on validation: {t_star} (break-even would be {COST_CALL / VALUE_CATCH:.3f})")
print(f"test months, checked once: {net_value(y[test].values, p_test, t_star):,.0f} EUR at {t_star} "
      f"vs {net_value(y[test].values, p_test, 0.5):,.0f} EUR at 0.5")

Calling costs little and a catch is worth a lot, so the best cut-off is low. With calibrated probabilities it
sits near the break-even point, cost ÷ value. That is the payoff of calibration.

### A forecast from probabilities
If the probabilities are calibrated, their **sum** over one night's arrivals is the expected number of
cancellations that night. That is exactly what an overbooking decision needs.

In [ ]:
nights = (df.loc[test].assign(p=p_test).query("hotel == 'City Hotel'")
          .groupby("arrival_date").agg(bookings=("p", "size"), expected=("p", "sum"), actual=("is_canceled", "sum")))
print(f"{len(nights)} nights · correlation expected vs actual: {nights['expected'].corr(nights['actual']):.2f}")
print(f"average per night: expected {nights['expected'].mean():.1f}, actual {nights['actual'].mean():.1f}")

In [ ]:
# @title Draw the forecast { display-mode: "form" }
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.plot(nights.index, nights["actual"], color=LAV, lw=1.5, label="actual cancellations")
ax.plot(nights.index, nights["expected"], color=ORANGE, lw=1.5, label="sum of predicted probabilities")
ax.set(ylabel="cancellations per night", title="City Hotel, test months: forecast vs reality"); ax.legend()
plt.show()

The forecast follows reality closely, night by night, and runs a little low on average. That gap is the drift
seen in section 2: in 2017 guests cancelled more than in the training years. In production this is what
**monitoring** watches for, and why models are retrained.

### From a forecast to an overbooking decision
Take one night: the hotel has `rooms` free rooms and `n` bookings due. How many extra rooms should it sell?
Each extra room earns about 110 EUR; each guest who arrives to a full hotel is walked, which costs about 250 EUR.
Simulating the night many times from the probabilities gives the expected profit for every choice.

In [ ]:
night = df.loc[test].assign(p=p_test).query("hotel == 'City Hotel' and arrival_date == @pd.Timestamp('2017-05-12')")
rooms, n = int(round(0.72 * len(night))), len(night)
sims = (np.random.default_rng(0).random((5000, n)) > night["p"].values).sum(axis=1)     # guests who show up
extra = np.arange(0, 41)
profit = [np.mean(110 * np.minimum(sims + k, rooms) - 250 * np.maximum(sims + k - rooms, 0)) for k in extra]
k_best = int(extra[np.argmax(profit)])
print(f"{n} bookings, {rooms} rooms · expected cancellations {night['p'].sum():.1f} · best: sell {k_best} extra rooms")
print(f"what actually happened: {(night['is_canceled'] == 0).sum()} guests showed up")

This is the engine of the app **Tonight's front desk**: you choose the number of extra rooms, the model makes
its own choice, then the real night is replayed.

---
# 🔍 8 · What drives a cancellation?

SHAP splits each prediction into contributions, one per feature, that add up to the model's output (in log-odds).
XGBoost computes these itself; `shap` draws them. Contributions of the one-hot columns are added back up to the
original feature, so each booking gets one number per question the hotel knows the answer to.

In [ ]:
sample = df.loc[test, FEATURES].sample(2000, random_state=0)
Xs = final[0].transform(sample)
contrib = final[-1].get_booster().predict(xgb.DMatrix(Xs), pred_contribs=True)[:, :-1]
names = final[0].get_feature_names_out()
original = [next(f for f in FEATURES if n.split("__", 1)[1] == f or n.split("__", 1)[1].startswith(f + "_")) for n in names]
by_feature = pd.DataFrame(contrib, columns=original).T.groupby(level=0).sum().T[FEATURES]
by_feature.abs().mean().sort_values(ascending=False).round(3).head(8)

In [ ]:
# @title Draw SHAP { display-mode: "form" }
show = sample.copy()
for c in CAT: show[c] = show[c].astype("category").cat.codes
shap.plots.beeswarm(shap.Explanation(by_feature.values, data=show.values, feature_names=FEATURES), max_display=10, show=False)
plt.title("Each dot is a booking · right = pushes towards cancellation"); plt.show()
i = int(np.argmax(final.predict_proba(sample)[:, 1]))
shap.plots.waterfall(shap.Explanation(by_feature.values[i], base_values=float(final[-1].get_booster().predict(
    xgb.DMatrix(Xs[i:i + 1]), pred_contribs=True)[0, -1]), data=sample.iloc[i].values, feature_names=FEATURES), max_display=8, show=False)
plt.title("The riskiest booking in the sample, explained"); plt.show()

Long lead times, non-refundable deposits and certain booking channels push towards cancellation; repeat guests
and short notice push away. Two warnings from session 09 still hold:

- **The model, not the world.** "Non-refundable deposit → cancels" sounds absurd. In this data, non-refundable
  bookings are mostly large group blocks from tour operators, which are often released in bulk. The feature
  marks a kind of booking; it does not make people cancel.
- **No causes.** Asking for deposits would not simply halve cancellations. That question needs an experiment
  (an A/B test), not a model.

---
# 📦 9 · Storing the model properly

A notebook is not a product. For an app to use the model, it needs a **stored artefact** that can be loaded
anywhere, and enough information to use it correctly:

| File | What it holds | Why |
|---|---|---|
| `model.joblib` | the whole pipeline: preprocessing + XGBoost | the app sends raw booking columns, not one-hot matrices |
| `config.json` | feature lists, library versions, metrics, periods, cut-off | the app checks its inputs and pins the same versions |
| `README.md` | the model card: purpose, data, performance, limits | the next person knows what it is for, and what not |

A pickled pipeline only loads reliably with the **same versions** of scikit-learn and XGBoost. That is why the
versions go into the config, and into the app's `requirements.txt`.

In [ ]:
import joblib, os
os.makedirs("hotel_model", exist_ok=True)
joblib.dump(final, "hotel_model/model.joblib")

config = {"model": "hotel-cancellation-xgb", "version": time.strftime("%Y-%m-%d"),
          "target": "is_canceled (1 = booking cancelled)",
          "numeric_features": NUM, "categorical_features": CAT,
          "categories": {c: sorted(df.loc[trainval, c].astype(str).unique().tolist()) for c in CAT},
          "periods": {"train": "2015-07-01 to 2016-11-30", "validation": "2016-12-01 to 2017-03-31",
                      "test": "2017-04-01 to 2017-08-31", "final fit": "train + validation"},
          "metrics_test": {"auc": round(roc_auc_score(y[test], p_test), 4), "log_loss": round(log_loss(y[test], p_test), 4),
                           "brier": round(brier_score_loss(y[test], p_test), 4)},
          "call_threshold": t_star, "costs_eur": {"call": COST_CALL, "caught_cancellation": VALUE_CATCH},
          "dropped_as_leakage": EXTRA,
          "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                       "xgboost": xgb.__version__, "pandas": pd.__version__, "numpy": np.__version__}}
json.dump(config, open("hotel_model/config.json", "w"), indent=2)

card = f"""---
license: cc-by-4.0
tags: [tabular-classification, xgboost, scikit-learn, teaching]
---
# Hotel cancellation model (teaching)

Predicts the probability that a hotel booking will be cancelled, from what is known **when the booking is made**.
Built in the Business Data Science programme, Aalborg University, session 10.

- **Data:** Antonio, Almeida & Nunes (2019), *Hotel booking demand datasets*, Data in Brief 22. Two Portuguese hotels, 2015–2017.
- **Model:** scikit-learn pipeline (imputation, scaling, one-hot) + XGBoost, tuned with Optuna on validation log loss.
- **Split:** by arrival date. Train to Nov 2016, validation Dec 2016–Mar 2017, test Apr–Aug 2017.
- **Test performance:** AUC {config['metrics_test']['auc']}, log loss {config['metrics_test']['log_loss']}, Brier {config['metrics_test']['brier']}.
- **Removed as leakage:** {', '.join(EXTRA)} (recorded or updated after the booking).
- **Limits:** two hotels, one country, 2015–2017; cancellations rose in 2017 and the model under-forecasts them slightly.
  Probabilities describe patterns in this data, not causes. Not for real pricing or staffing decisions.

Load with `joblib.load("model.joblib")` using the versions in `config.json`.
"""
open("hotel_model/README.md", "w").write(card)

# The app replays real nights from the test months, which the model's training never saw
cols = FEATURES + ["arrival_date", "is_canceled"]
df.loc[test, cols].to_parquet("hotel_model/test_bookings.parquet", index=False)
print(sorted(os.listdir("hotel_model")), "·", {k: v for k, v in config["versions"].items() if k != "python"})

### Share it on the Hugging Face Hub (optional)

The Hub is a place for models the way GitHub is for code: versioned, with a model card, loadable from anywhere.
The app for this session loads the course's copy from
[AAUBS/hotel-cancellation-model](https://huggingface.co/AAUBS/hotel-cancellation-model).
To publish your own: create a free account, make a token with *write* access (Settings → Access Tokens), then run
the cell below and paste the token when asked.

In [ ]:
PUBLISH = False          # set to True to upload your own copy
if PUBLISH:
    from huggingface_hub import HfApi, login
    login()                                                        # paste your write token
    api = HfApi()
    repo = f"{api.whoami()['name']}/hotel-cancellation-model"
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(folder_path="hotel_model", repo_id=repo, commit_message="Session 10 model")
    print("uploaded:", f"https://huggingface.co/{repo}")

---
# 📋 Wrap-up

- **Time:** split by date when the model will be used on the future; a random split flatters the model.
- **Leakage:** ask of every column whether it exists when the prediction is made. Here six did not.
- **Ensembles:** bagging averages many overfitting trees (less variance); boosting adds small trees that fix
  the remaining errors (less bias). On business tables, tuned boosting is usually the strongest model.
- **Optimisation:** tune on validation data with a proper score; Optuna searches smarter than a grid.
- **Calibration:** probabilities must mean what they say before they are added up or compared with costs.
  Class weights distort them.
- **Decisions:** thresholds from costs, forecasts from sums of probabilities, overbooking from simulation.
- **Artefacts:** store the whole pipeline, its config and versions, and a model card.

**Next:** the slides on how deployment works, then the app
[Tonight's front desk](https://tonights-front-desk.streamlit.app): play it, then fork
[its code](https://github.com/aaubs/tonights-front-desk) and make it yours.